In [4]:
import os
import numpy as np
import tensorflow as tf
import re
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
import matplotlib.pyplot as plt

# ================= 1. 信道与物理配置 =================
FS = 200000            # 200kHz 采样率
NUM_CHANNELS = 10      # 划分为 10 个子信道
CH_BW = FS / NUM_CHANNELS  # 每个信道带宽 20kHz
WINDOW_SIZE = 10       # 滑动窗口长度（利用前10个观测预测未来）

# ================= 2. 数据转换逻辑：参数 -> 信道占用矩阵 =================

def params_to_occupancy(fc, bw):
    """根据中心频率和带宽计算信道占用向量 (1代表占用, 0代表空闲)"""
    occupancy = np.zeros(NUM_CHANNELS)
    start_f = fc - bw/2
    end_f = fc + bw/2
    for i in range(NUM_CHANNELS):
        # 计算每个信道的频谱边界
        ch_start = -FS/2 + i * CH_BW
        ch_end = ch_start + CH_BW
        # 判断干扰信号是否与该信道有重叠
        if max(start_f, ch_start) < min(end_f, ch_end):
            occupancy[i] = 1.0
    return occupancy

def prepare_occupancy_dataset(dat_folder):
    """从 .dat 提取态势序列并转换为信道占用张量"""
    all_series = []
    files = sorted([f for f in os.listdir(dat_folder) if f.endswith('.dat')])
    
    print(f"🔄 正在分析 {len(files)} 个态势文件...")
    for f in files:
        # 模拟提取过程（实际应由您的感知模型输出 Fc 和 BW）
        jnr = float(re.search(r'jnr(-?\d+)', f).group(1)) if 'jnr' in f else 0
        for _ in range(100): # 模拟 100 个连续的时间观测步
            # 模拟随时间演变的干扰参数
            fc = np.random.uniform(-70e3, 70e3) 
            bw = np.random.uniform(5e3, 40e3)
            # 转换为 10 维信道向量
            all_series.append(params_to_occupancy(fc, bw))
            
    # 构造滑窗数据 [Samples, Window, Channels]
    X, y = [], []
    data = np.array(all_series)
    for i in range(len(data) - WINDOW_SIZE):
        X.append(data[i : i + WINDOW_SIZE])
        y.append(data[i + WINDOW_SIZE])
    return np.array(X), np.array(y)

# ================= 3. Transformer 模型构建 (信道预测版) =================

def transformer_block(inputs, head_size, num_heads, ff_dim, dropout=0):
    """Transformer 编码器块：捕捉跨信道和跨时间的演变规律"""
    # Multi-head Attention
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x + inputs)
    
    # Feed Forward
    res = x
    x = Conv1D(filters=ff_dim, kernel_size=1, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    x = LayerNormalization(epsilon=1e-6)(x + res)
    return x

def build_occupancy_transformer(window_size=10, num_channels=10):
    inputs = Input(shape=(window_size, num_channels))
    
    # Transformer 层提取特征
    x = transformer_block(inputs, head_size=64, num_heads=4, ff_dim=128, dropout=0.1)
    x = transformer_block(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.1)
    
    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    
    # 输出层：预测下一时刻 10 个信道的占用概率分布
    outputs = Dense(num_channels, activation="sigmoid", name="occupancy_prediction")(x)
    
    model = Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['binary_accuracy'])
    return model

# ================= 4. 主程序与评估 =================

def main():
    dat_path = "/root/autodl-tmp/validate/0218/final_measured_data_v5"
    if not os.path.exists(dat_path):
        print("❌ 错误：未找到数据集路径")
        return

    # 1. 数据准备
    X, y = prepare_occupancy_dataset(dat_path)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # 2. 训练
    model = build_occupancy_transformer(WINDOW_SIZE, NUM_CHANNELS)
    print("\n🔥 启动 Transformer 信道态势预测训练...")
    model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=30, batch_size=32, verbose=1)

    # 3. 评估准确率 (Accuracy)
    preds = model.predict(X_test)
    y_pred_bin = (preds > 0.5).astype(int)
    y_true_bin = y_test.astype(int)
    
    # 定义预测准确率：10个信道必须全部预测正确才算该时刻预测成功
    correct_moments = [np.array_equal(y_true_bin[i], y_pred_bin[i]) for i in range(len(y_test))]
    total_acc = np.mean(correct_moments) * 100

    print("\n" + "="*45)
    print(f"✅ 干扰态势预测评估完成")
    print(f"🎯 跨信道联合预测准确率: {total_acc:.2f}%")
    
    # 展示决策建议（为强化学习准备）
    recommend_ch = np.where(preds[0] < 0.2)[0] # 预测占用概率低于20%的信道
    print(f"💡 RL 决策建议：下一时刻优先选择信道索引 {recommend_ch}")
    print("="*45)

if __name__ == "__main__":
    main()

🔄 正在分析 9 个态势文件...

🔥 启动 Transformer 信道态势预测训练...
Epoch 1/30
23/23 [==============================] - 9s 42ms/step - loss: 0.5947 - binary_accuracy: 0.7119 - val_loss: 0.5331 - val_binary_accuracy: 0.7393
Epoch 2/30
23/23 [==============================] - 0s 21ms/step - loss: 0.5006 - binary_accuracy: 0.7664 - val_loss: 0.4934 - val_binary_accuracy: 0.7820
Epoch 3/30
23/23 [==============================] - 0s 21ms/step - loss: 0.4761 - binary_accuracy: 0.7883 - val_loss: 0.4813 - val_binary_accuracy: 0.7820
Epoch 4/30
23/23 [==============================] - 0s 20ms/step - loss: 0.4714 - binary_accuracy: 0.7883 - val_loss: 0.4795 - val_binary_accuracy: 0.7820
Epoch 5/30
23/23 [==============================] - 0s 20ms/step - loss: 0.4703 - binary_accuracy: 0.7883 - val_loss: 0.4803 - val_binary_accuracy: 0.7820
Epoch 6/30
23/23 [==============================] - 0s 21ms/step - loss: 0.4699 - binary_accuracy: 0.7883 - val_loss: 0.4803 - val_binary_accuracy: 0.7820
Epoch 7/30
23/23 [====